In [25]:
import requests
base_url = "http://localhost:8080"
request_data = {
    "company_description": "File creator",
    "company_domain": "files.com",
    "hiring_needs": "We are looking for someone who can store an create files",
    "specific_benefits": "Remote work, flexible hours, stock options",
    "agents": [
        {
            "role": "File maker",
            "goal": "create new files and save them",
            "backstory": "You are a 40 year old administrator that creates files and stores them",
            "class_tools": [{"name": "save_file", "description": "Tool for version control"}],
            "tools": [
                # {"name": "file_tool", "description": "Tool for storing files"},
            ],
        }
    ],
    "tasks": [
        {
            "description": "Make files",
            "expected_output": "Create a .txt file that says hello world and write it to the path /home/egbert",
            "agent_role": "File maker"
        },
    ]
}

response = requests.post(f"{base_url}/agent/crewai", json=request_data)
assert response.status_code == 200
data = response.json()
print(data)

{'message': 'Crew created', 'crew_id': {'id': 'd7a5e085-e03d-4966-892e-bd955d0c7325', 'create_crew': {'company_description': 'File creator', 'company_domain': 'files.com', 'hiring_needs': 'We are looking for someone who can store an create files', 'specific_benefits': 'Remote work, flexible hours, stock options', 'agents': [{'role': 'File maker', 'goal': 'create new files and save them', 'backstory': 'You are a 40 year old administrator that creates files and stores them', 'tools': [], 'class_tools': [{'name': 'save_file', 'description': 'Tool for version control'}]}], 'tasks': [{'description': 'Make files', 'expected_output': 'Create a .txt file that says hello world and write it to the path /home/egbert', 'agent_role': 'File maker'}]}, 'created_at': '2024-07-24T23:20:46.236322', 'updated_at': None}, 'curried_proxy_id': 'c86e18ea-2204-4e65-8139-8d70ff35e748'}


In [26]:

from src.server.models.message import Message

base_url = "http://localhost:8080"

# Create tasksmanager
tasksmanager_data = {
    "name": "File expert",
    "model": "gpt-4",
    "description": "make files",
}

tasksmanager_response = requests.post(f"{base_url}/tasksmanager", json=tasksmanager_data)
assert tasksmanager_response.status_code == 201
tasksmanager_id = tasksmanager_response.json()["id"]

# Create task
system_message = Message(
    role="system",
    content="You create new files based on input"
)

task_data = {
    "name": "Solar System Questions",
    "description": "Answer questions about the solar system, alway anser distance in Miles.",
    "tasksmanager_id": tasksmanager_id,
    "messages": [system_message.dict()],
    "crewai_info_id": data['crew_id']['id'],
    "curried_proxy_id": data['curried_proxy_id'],
}

task_response = requests.post(f"{base_url}/task", json=task_data)
assert task_response.status_code == 201
task_id = task_response.json()["id"]

# Run task with user message
user_message = Message(
    role="user",
    content="write a file to /home/egbert/hello.txt that says hello world"
)
run_task = {
    'caller_id': "UnitTestCaller",
    'messages': [user_message.dict()]
}

run_task_response = requests.post(f"{base_url}/task/{tasksmanager_id}/{task_id}/run", json=run_task)
print(run_task_response)

<Response [200]>
